In [ ]:
# --- bootstrap: anchor to the repository root, wherever this notebook was opened from ---
# Notebooks live two levels deep under notebooks/, so the cwd-relative path logic below needs the
# root established first. Keyed on pytest.ini, which is not tied to any folder-naming decision.
import os
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

# Report Visual Plan: Financial-News Sentiment and Short-Term Market Direction

**Academic research only. Not investment advice.**

This notebook turns Team Alpha Signal's project proposal into four report-ready figures. The visual story is deliberately sequenced: first establish data coverage, then verify sentiment variation, then show the model-free sentiment-return pattern, and finally test whether sentiment improves out-of-sample prediction. A null result remains informative.

The final report focuses on one consistent project scope:

1. **News signal:** global macroeconomic news from WRDS RavenPack, scored using RavenPack’s built-in sentiment and FinBERT.
2. **Prediction universe:** the 11 SPDR sector ETFs representing the major S&P 500 sectors.
3. **Prediction target:** whether each sector ETF has a positive return during the next trading session.
4. **Main comparison:** market-only features versus RavenPack sentiment, FinBERT sentiment, and their combination.

The individual-stock S&P 500 analysis remains an exploratory extension and is not required for the final MVP.

No raw RavenPack records, headlines, or licensed WRDS exports are saved by this notebook.

## Four-Figure Story

| Figure | Role in the report | Decision it supports |
| --- | --- | --- |
| 1. Data coverage and sentiment over time | Establish the integrity and temporal behavior of the news signal. | Is the sample sufficiently complete and stable to analyze? |
| 2. Distribution of RavenPack sentiment scores | Verify that the sentiment feature varies meaningfully. | Is there enough signal variation to justify modeling? |
| 3. Future returns by sentiment quantile | Show the raw, model-free association before fitting a classifier. | Is there a descriptive relationship worth modeling? |
| 4. Incremental out-of-sample performance | Compare a market-only baseline against a sentiment-augmented model. | Does sentiment add predictive information beyond market features? |

Figures 1 and 3 can be rendered from derived daily aggregates. Figure 2 activates once the data pipeline writes a pre-binned sentiment-count summary, and Figure 4 activates after the modeling workflow writes derived evaluation files. The notebook intentionally does not train a model.

The broad sentiment-bucket return chart and confusion-matrix error profile remain later in the notebook as supplemental diagnostics. This keeps the main report focused on four figures.

## Figure 1: Data Coverage and Sentiment Over Time

- **Variables:** trading session date; RavenPack `event_record_count`; `unique_story_count`; and `mean_event_sentiment_score`.
- **Visualization:** two aligned weekly time-series panels: news volume and mean sentiment, with a zero reference line on the sentiment panel.
- **Takeaway:** readers should be able to see whether the news supply is stable across the sample and whether sentiment changes through identifiable market regimes. This validates that later results are not driven by an empty or unusually thin portion of the data.

This is a data-quality and context figure, not evidence that sentiment predicts returns.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")

# The notebook can be started from the repository root or from data/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    if (REPO_ROOT.parent / "data").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise FileNotFoundError("Run this notebook from the repository root or data/.")

DATA_DIR = REPO_ROOT / "data"
NEWS_PATH = DATA_DIR / "news_daily_df.csv"
MARKET_PATH = DATA_DIR / "market_daily_df.csv"
MODEL_OUTPUT_DIR = REPO_ROOT / "outputs"
METRICS_PATH = MODEL_OUTPUT_DIR / "walk_forward_metrics.csv"
PREDICTIONS_PATH = MODEL_OUTPUT_DIR / "holdout_predictions.csv"
SENTIMENT_DISTRIBUTION_PATH = DATA_DIR / "sentiment_score_distribution_df.csv"
ANALYSIS_START = pd.Timestamp("2020-01-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")

if not NEWS_PATH.exists() or not MARKET_PATH.exists():
    raise FileNotFoundError("Expected derived daily CSVs in data/. No raw WRDS data is read.")

news_daily_df = pd.read_csv(NEWS_PATH, parse_dates=["session_date"])
market_daily_df = pd.read_csv(MARKET_PATH, parse_dates=["session_date"])
news_daily_df = news_daily_df.loc[news_daily_df["session_date"].between(ANALYSIS_START, ANALYSIS_END)].copy()
market_daily_df = market_daily_df.loc[market_daily_df["session_date"].between(ANALYSIS_START, ANALYSIS_END)].copy()
asset_column = "asset" if "asset" in market_daily_df.columns else "ticker"

planned_index_assets = {"SPX", "NASDAQ_COMPOSITE", "DOW_JONES_INDUSTRIALS", "SOX"}
observed_assets = set(market_daily_df[asset_column].dropna().astype(str).unique())
if planned_index_assets.issubset(observed_assets):
    chart_scope = "WRDS market-index prototype"
else:
    chart_scope = "current local panel (chart template only)"
    print(
        "Note: the local market CSV is not the planned four-index panel. "
        "Figures 1 and 2 are rendered only to test the presentation pattern; "
        "final report claims must use the true index or firm-level data."
    )

print(f"News sessions: {len(news_daily_df):,}")
print(f"Market rows: {len(market_daily_df):,}")
print(f"Analysis window: {ANALYSIS_START.date()} to {ANALYSIS_END.date()}")
print(f"Observed targets: {sorted(observed_assets)}")


In [ ]:
weekly_news_df = (
    news_daily_df.set_index("session_date")
    .sort_index()
    .resample("W-FRI")
    .agg(
        event_record_count=("event_record_count", "sum"),
        unique_story_count=("unique_story_count", "sum"),
        mean_event_sentiment_score=("mean_event_sentiment_score", "mean"),
    )
    .dropna(how="all")
)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
axes[0].plot(
    weekly_news_df.index,
    weekly_news_df["event_record_count"],
    color="#1f77b4",
    linewidth=1.4,
    label="Event records",
)
axes[0].plot(
    weekly_news_df.index,
    weekly_news_df["unique_story_count"],
    color="#6baed6",
    linewidth=1.2,
    label="Unique stories",
)
axes[0].set_ylabel("Weekly count")
axes[0].set_title("Figure 1. News coverage and mean sentiment over time")
axes[0].legend(frameon=False, ncol=2)

axes[1].plot(
    weekly_news_df.index,
    weekly_news_df["mean_event_sentiment_score"],
    color="#2e8b57",
    linewidth=1.4,
)
axes[1].axhline(0, color="#4d4d4d", linewidth=0.8)
axes[1].set_ylabel("Mean event sentiment")
axes[1].set_xlabel("Trading week")
fig.text(
    0.01,
    -0.02,
    f"Scope: {chart_scope}. Weekly summaries are derived from the daily RavenPack aggregate.",
    ha="left",
    fontsize=9,
)
plt.show()


## Supplemental Diagnostic: Forward Returns by Broad Sentiment Bucket

- **Variables:** daily sentiment bucket (`negative`, `neutral`, `positive`); equal-weighted forward 1-day return; and equal-weighted forward 5-day cumulative return.
- **Visualization:** paired bar charts of mean forward return in basis points, with 95% normal-approximation confidence intervals and a zero reference line. Each session contributes once after aggregating across targets, so a market-wide macro signal is not counted repeatedly.
- **Takeaway:** readers should see the direction, economic size, and uncertainty of the raw sentiment-return relationship before any classifier is introduced. Overlapping intervals or an inconsistent pattern are a meaningful descriptive null result, not a failure.

This figure is a sanity check. It does not establish causality or incremental predictive value.

In [ ]:
required_news_columns = {"session_date", "sentiment_bucket"}
required_return_columns = {"session_date", "fwd_1d_return", "fwd_5d_return"}
missing_columns = (required_news_columns - set(news_daily_df.columns)) | (
    required_return_columns - set(market_daily_df.columns)
)
if missing_columns:
    raise ValueError(f"Figure 2 is missing required columns: {sorted(missing_columns)}")

session_return_df = (
    market_daily_df[["session_date", "fwd_1d_return", "fwd_5d_return"]]
    .merge(news_daily_df[["session_date", "sentiment_bucket"]], on="session_date", how="inner", validate="many_to_one")
    .groupby(["session_date", "sentiment_bucket"], as_index=False)[["fwd_1d_return", "fwd_5d_return"]]
    .mean()
)

bucket_order = ["negative", "neutral", "positive"]
bucket_colors = {"negative": "#c0392b", "neutral": "#7f8c8d", "positive": "#2e8b57"}
figure_two_rows = []
for horizon, horizon_label in [("fwd_1d_return", "Next trading day"), ("fwd_5d_return", "Next five trading days")]:
    summary = (
        session_return_df.groupby("sentiment_bucket")[horizon]
        .agg(n="count", mean="mean", std="std")
        .reindex(bucket_order)
        .dropna(subset=["n"])
        .reset_index()
    )
    summary["ci_95"] = 1.96 * summary["std"] / np.sqrt(summary["n"])
    summary["horizon"] = horizon_label
    figure_two_rows.append(summary)

figure_two_summary_df = pd.concat(figure_two_rows, ignore_index=True)
display(figure_two_summary_df[["horizon", "sentiment_bucket", "n", "mean", "ci_95"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True, constrained_layout=True)
for axis, horizon_label in zip(axes, ["Next trading day", "Next five trading days"]):
    plot_df = figure_two_summary_df.query("horizon == @horizon_label").copy()
    x_positions = np.arange(len(plot_df))
    axis.bar(
        x_positions,
        plot_df["mean"] * 10_000,
        yerr=plot_df["ci_95"] * 10_000,
        capsize=4,
        color=[bucket_colors[bucket] for bucket in plot_df["sentiment_bucket"]],
        edgecolor="white",
    )
    axis.axhline(0, color="#4d4d4d", linewidth=0.8)
    axis.set_xticks(x_positions, plot_df["sentiment_bucket"].str.title())
    axis.set_title(horizon_label)
    axis.set_xlabel("Sentiment bucket")
axes[0].set_ylabel("Mean forward return (basis points)")
fig.suptitle("Supplemental diagnostic. Model-free forward returns by broad sentiment bucket", y=1.03)
plt.show()


## Figure 4: Incremental Out-of-Sample Performance

- **Variables:** model feature set (`baseline market features` versus `baseline + sentiment`); chronological validation fold; and held-out AUC-ROC, F1 score, and directional accuracy.
- **Visualization:** a three-panel point-and-interval plot. Each point is the average held-out score across walk-forward folds; the interval summarizes variation across folds.
- **Takeaway:** this is the main decision figure. It answers whether sentiment provides incremental predictive value after recent returns, sector momentum, and other market controls are included. The report should claim added value only when the sentiment model improves consistently across future validation folds, not merely on one split.

The model workflow should write the derived file `outputs/walk_forward_metrics.csv` with columns `model`, `fold`, `metric`, and `score`. Store metrics only; do not export restricted event-level data.

In [ ]:
if not METRICS_PATH.exists():
    print(
        "Figure 4 is planned but not rendered. After walk-forward evaluation, write "
        f"{METRICS_PATH.relative_to(REPO_ROOT)} with model, fold, metric, and score columns."
    )
else:
    metrics_df = pd.read_csv(METRICS_PATH)
    required_metric_columns = {"model", "fold", "metric", "score"}
    missing_metric_columns = required_metric_columns - set(metrics_df.columns)
    if missing_metric_columns:
        raise ValueError(f"Metrics file is missing columns: {sorted(missing_metric_columns)}")

    metric_labels = {
        "auc_roc": "AUC-ROC",
        "f1": "F1 score",
        "directional_accuracy": "Directional accuracy",
    }
    plot_metrics = [metric for metric in metric_labels if metric in set(metrics_df["metric"])]
    if not plot_metrics:
        raise ValueError(f"No expected metrics found. Expected one or more of: {list(metric_labels)}")

    models = list(metrics_df["model"].dropna().unique())
    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5.2 * len(plot_metrics), 4.8), sharey=True, constrained_layout=True)
    if len(plot_metrics) == 1:
        axes = [axes]

    for axis, metric in zip(axes, plot_metrics):
        metric_df = metrics_df.loc[metrics_df["metric"] == metric]
        summary = metric_df.groupby("model")["score"].agg(mean="mean", std="std", n="count").reindex(models)
        ci_95 = 1.96 * summary["std"].fillna(0) / np.sqrt(summary["n"])
        x_positions = np.arange(len(summary))
        axis.errorbar(x_positions, summary["mean"], yerr=ci_95, fmt="o", capsize=5, color="#1f77b4")
        axis.set_xticks(x_positions, summary.index, rotation=20, ha="right")
        axis.set_title(metric_labels[metric])
        axis.set_ylim(0, 1)
    axes[0].set_ylabel("Held-out score")
    fig.suptitle("Figure 4. Walk-forward performance: baseline versus sentiment-augmented model", y=1.03)
    plt.show()


## Supplemental Diagnostic: Held-Out Error Profile

- **Variables:** model (`baseline market features` versus `baseline + sentiment`); true forward direction (`negative` or `positive`); and predicted forward direction.
- **Visualization:** two side-by-side confusion-matrix heatmaps using the same class order and color scale.
- **Takeaway:** readers should see whether an improvement in AUC, F1, or directional accuracy corresponds to fewer false positives or false negatives. This protects against a headline metric improving because of a threshold or class-balance artifact.

The model workflow should write `outputs/holdout_predictions.csv` with columns `model`, `y_true`, and `y_pred`. Use predictions from a strictly future holdout period only.

In [ ]:
if not PREDICTIONS_PATH.exists():
    print(
        "Supplemental error-profile figure is planned but not rendered. After final held-out evaluation, write "
        f"{PREDICTIONS_PATH.relative_to(REPO_ROOT)} with model, y_true, and y_pred columns."
    )
else:
    predictions_df = pd.read_csv(PREDICTIONS_PATH)
    required_prediction_columns = {"model", "y_true", "y_pred"}
    missing_prediction_columns = required_prediction_columns - set(predictions_df.columns)
    if missing_prediction_columns:
        raise ValueError(f"Predictions file is missing columns: {sorted(missing_prediction_columns)}")

    models = list(predictions_df["model"].dropna().unique())
    class_labels = sorted(set(predictions_df["y_true"].dropna()) | set(predictions_df["y_pred"].dropna()))
    if not class_labels:
        raise ValueError("Predictions file has no non-null class labels.")

    matrices = []
    for model in models:
        model_df = predictions_df.loc[predictions_df["model"] == model]
        matrix = pd.crosstab(model_df["y_true"], model_df["y_pred"]).reindex(index=class_labels, columns=class_labels, fill_value=0)
        matrices.append((model, matrix))

    color_max = max(matrix.to_numpy().max() for _, matrix in matrices)
    fig, axes = plt.subplots(1, len(matrices), figsize=(5.2 * len(matrices), 4.7), constrained_layout=True)
    if len(matrices) == 1:
        axes = [axes]

    for axis, (model, matrix) in zip(axes, matrices):
        image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=color_max)
        for row_index, row_label in enumerate(class_labels):
            for column_index, column_label in enumerate(class_labels):
                axis.text(column_index, row_index, matrix.loc[row_label, column_label], ha="center", va="center")
        axis.set_xticks(range(len(class_labels)), class_labels)
        axis.set_yticks(range(len(class_labels)), class_labels)
        axis.set_xlabel("Predicted direction")
        axis.set_ylabel("True direction")
        axis.set_title(str(model))
    fig.colorbar(image, ax=axes, shrink=0.82, label="Held-out observations")
    fig.suptitle("Supplemental diagnostic. Held-out prediction errors by model", y=1.03)
    plt.show()


## Figure 2: Distribution of RavenPack Sentiment Scores

- **Variables visualized:** RavenPack event sentiment score, optionally grouped with RavenPack's documented negative, neutral, and positive categories.
- **Visualization type:** histogram or density plot of event-level scores, with a zero reference line.
- **Takeaway point:** this figure shows whether the sentiment variable has enough variation to be useful. If most events cluster near neutral, the signal may be weaker. A clear spread across negative and positive values makes sentiment more promising as a later prediction feature.

The daily news gold table contains the **mean** event sentiment by session, so it cannot support an honest event-level distribution. The code below therefore expects a non-restricted, pre-binned count table rather than raw RavenPack events.

In [ ]:
if not SENTIMENT_DISTRIBUTION_PATH.exists():
    print(
        "Figure 2 is planned but not rendered. Create a derived, pre-binned "
        f"table at {SENTIMENT_DISTRIBUTION_PATH.relative_to(REPO_ROOT)} with columns "
        "sentiment_score_bin and event_record_count. Do not save raw RavenPack events."
    )
else:
    distribution_df = pd.read_csv(SENTIMENT_DISTRIBUTION_PATH)
    required_distribution_columns = {"sentiment_score_bin", "event_record_count"}
    missing_distribution_columns = required_distribution_columns - set(distribution_df.columns)
    if missing_distribution_columns:
        raise ValueError(
            f"Distribution file is missing columns: {sorted(missing_distribution_columns)}"
        )

    plot_distribution_df = distribution_df.copy()
    plot_distribution_df["sentiment_score_bin"] = pd.to_numeric(
        plot_distribution_df["sentiment_score_bin"], errors="raise"
    )
    plot_distribution_df = plot_distribution_df.sort_values("sentiment_score_bin")
    bin_steps = np.diff(plot_distribution_df["sentiment_score_bin"].unique())
    bin_width = 0.8 * np.median(bin_steps) if len(bin_steps) else 0.05

    fig, axis = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
    axis.bar(
        plot_distribution_df["sentiment_score_bin"],
        plot_distribution_df["event_record_count"],
        width=bin_width,
        color="#4c78a8",
        edgecolor="white",
    )
    axis.axvline(0, color="#4d4d4d", linewidth=0.9)
    axis.set_title("Figure 2. Distribution of RavenPack event sentiment scores")
    axis.set_xlabel("RavenPack event sentiment score (binned)")
    axis.set_ylabel("Event count")
    plt.show()


## Figure 3: Future Returns by Sentiment Quantile

- **Variables visualized:** five sentiment quantile groups (`Lowest 20%`, `Low`, `Middle`, `High`, `Highest 20%`); average next-day return; average five-day cumulative return; and next-day positive-return rate.
- **Visualization type:** grouped small-multiple bar charts by sentiment quantile, with uncertainty intervals and a zero line for return outcomes.
- **Takeaway point:** this figure communicates the main preliminary relationship between news sentiment and market movement. It shows whether higher sentiment groups tend to precede stronger short-term returns or a higher positive-return rate. It is exploratory evidence, not formal proof of predictability.

For the macro/index prototype, form quantiles on the daily aggregate sentiment score and collapse returns to one equal-weighted market observation per session.For the final sector-ETF analysis, form daily FinBERT and RavenPack sentiment quantiles and compare subsequent returns across the 11 sector ETFs.

In [ ]:
quantile_required_news_columns = {"session_date", "mean_event_sentiment_score"}
quantile_required_market_columns = {"session_date", "fwd_1d_return", "fwd_5d_return"}
quantile_missing_columns = (quantile_required_news_columns - set(news_daily_df.columns)) | (
    quantile_required_market_columns - set(market_daily_df.columns)
)
if quantile_missing_columns:
    raise ValueError(f"Quantile figure is missing required columns: {sorted(quantile_missing_columns)}")

# Collapse the repeated macro signal to one equal-weighted market return per session.
quantile_session_df = (
    market_daily_df.groupby("session_date", as_index=False)[["fwd_1d_return", "fwd_5d_return"]]
    .mean()
    .merge(
        news_daily_df[["session_date", "mean_event_sentiment_score"]],
        on="session_date",
        how="inner",
        validate="one_to_one",
    )
    .dropna(subset=["mean_event_sentiment_score"])
    .sort_values("session_date")
    .reset_index(drop=True)
)

quantile_labels = ["Lowest 20%", "Low", "Middle", "High", "Highest 20%"]
if len(quantile_session_df) < len(quantile_labels):
    raise ValueError("At least five aligned sessions are required to form sentiment quantiles.")

# Ranking resolves duplicate daily scores while preserving equal-sized exploratory groups.
score_rank = quantile_session_df["mean_event_sentiment_score"].rank(method="first")
quantile_session_df["sentiment_quantile"] = pd.qcut(
    score_rank, q=len(quantile_labels), labels=quantile_labels
)
quantile_session_df["fwd_1d_positive"] = (quantile_session_df["fwd_1d_return"] > 0).astype(float)

quantile_summary_df = (
    quantile_session_df.groupby("sentiment_quantile", observed=False)
    .agg(
        n_1d=("fwd_1d_return", "count"),
        mean_1d=("fwd_1d_return", "mean"),
        std_1d=("fwd_1d_return", "std"),
        n_5d=("fwd_5d_return", "count"),
        mean_5d=("fwd_5d_return", "mean"),
        std_5d=("fwd_5d_return", "std"),
        positive_rate=("fwd_1d_positive", "mean"),
    )
    .reindex(quantile_labels)
)
quantile_summary_df["ci_1d"] = 1.96 * quantile_summary_df["std_1d"] / np.sqrt(quantile_summary_df["n_1d"])
quantile_summary_df["ci_5d"] = 1.96 * quantile_summary_df["std_5d"] / np.sqrt(quantile_summary_df["n_5d"])
quantile_summary_df["ci_positive_rate"] = 1.96 * np.sqrt(
    quantile_summary_df["positive_rate"] * (1 - quantile_summary_df["positive_rate"]) / quantile_summary_df["n_1d"]
)
display(quantile_summary_df)

quantile_colors = ["#c0392b", "#d6604d", "#7f8c8d", "#5aae61", "#2e8b57"]
plot_specs = [
    ("mean_1d", "ci_1d", "Average next-day return (basis points)", 10_000, True),
    ("mean_5d", "ci_5d", "Average five-day return (basis points)", 10_000, True),
    ("positive_rate", "ci_positive_rate", "Next-day positive-return rate (%)", 100, False),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
x_positions = np.arange(len(quantile_summary_df))
for axis, (value_column, error_column, y_label, multiplier, zero_line) in zip(axes, plot_specs):
    axis.bar(
        x_positions,
        quantile_summary_df[value_column] * multiplier,
        yerr=quantile_summary_df[error_column].fillna(0) * multiplier,
        capsize=4,
        color=quantile_colors,
        edgecolor="white",
    )
    if zero_line:
        axis.axhline(0, color="#4d4d4d", linewidth=0.8)
    axis.set_xticks(x_positions, quantile_labels, rotation=20, ha="right")
    axis.set_ylabel(y_label)
fig.suptitle("Figure 3. Future returns by daily sentiment quantile", y=1.03)
plt.show()


## Reporting Guardrails

- Label Figures 1 and 2 as **descriptive**. They cannot show that sentiment causes returns or that it survives market controls.
- Keep the macro/index prototype separate from the final company-specific S&P 500 analysis. Do not pool them or describe one as evidence for the other.
- Construct all sentiment features using news available by the prior market close. Use time-ordered walk-forward validation and a final future holdout for Figures 3 and 4.
- Report confidence intervals, the number of observations, and the exact target horizon. Show the null result plainly if intervals overlap or the augmented model does not outperform.
- Keep the language academic: the figures evaluate a research hypothesis and are not trading recommendations.
- Save only derived summaries, model metrics, and visualizations. Do not commit raw RavenPack data, article text, headlines, or WRDS exports.